# Building a DECAT Dataset

This notebook is a helper tool for constructing an `ObsTable` compatible data set from the DECAT visit data in [https://github.com/gnarayan/decat_pointings](https://github.com/gnarayan/decat_pointings). It is based on the [DECAT_qcinv_summary](https://github.com/gnarayan/decat_pointings/blob/main/notebooks/DECAT_qcinv_summary.ipynb) notebook by Melissa Graham.

To run this notebook, users need to first `git clone` the data repository locally and set the `DATA_DIR` to the corresponding local directory.


## Locating the qcinv files

The DECAT data is stored in a nest series of directories.

In [ ]:
import glob
import numpy as np

# The default is to assume the lightcurvelynx and decat_pointings diectories
# are in the same parent directory. You may need to change this parameter if
# your directory structure is different.
DATA_DIR = "../../decat_pointings"

# Get the list of all .qcinv files in the data directory sorted by path.
filenames = sorted(glob.glob(f"{DATA_DIR}/*/*/*.qcinv"))
print(f"Found {len(filenames)} .qcinv files.")

To simplify processing, we add a helper function that will compute the MJD from the file's path and the ut column of the data (which is in the form "HH:MM").

In [ ]:
from astropy.time import Time
from pathlib import Path


def mjd_from_filename(file_path, ut=None):
    """Extract the base MJD of the pointing from the file's path.

    Parameters
    ----------
    file_path : str
        The path to the .qcinv file in the form:
        <PREFIX>/<YYYYMMDD>.qcinv
    ut : str, optional
        The UT time to use for the MJD calculation in the format "HH:MM:SS".
        If None, the time will be set to 00:00:00.
    """
    if ut is None or ut.strip() == "":
        ut = "00:00:00.00"
    elif len(ut.split(":")) == 2:
        ut += ":00.00"
    if len(ut.split(":")) != 3:
        raise ValueError(f"UT time must be in the format 'HH:MM:SS' or 'HH:MM', got '{ut}'")

    file_name = Path(file_path).stem  # Get the file name without extension
    # Check that the file name has the expected format of YYYYMMDD
    if len(file_name) < 8 or not file_name[:8].isdigit():
        raise ValueError(f"File name '{file_name}' does not start with a valid date in the format YYYYMMDD")

    isot_str = f"{file_name[0:4]}-{file_name[4:6]}-{file_name[6:8]}T{ut}"
    mjd = Time(isot_str, format="isot", scale="utc").mjd
    return mjd

Next we build a helper function to parse a single qcinv file. The data is in fixed ASCII format, so we need to read each line and extract the columns manually (although maybe there is a good tool for this):

  * **expid** (characters 0-5) - An experiment ID [We do not use this]
  * **ra** (characters 6-11) - Right Ascension in degrees
  * **dec** (characters 12-17) - Declination in degrees
  * **ut** (characters 19-23) - ut time as a string "HH:MM"
  * **fil** (characters 24-27) - filter name as a string
  * **time** (characters 28-31) - Exposure time in seconds
  * **secz** (characters 32-36) - Airmass
  * **psf** (characters 37-42) - PSF (blank for no data)
  * **sky** (characters 43-48) - sky (blank for no data)
  * **cloud** (characters 49-54) - cloud (blank for no data)
  * **teff** (characters 55-60) - teff (blank for no data)
  * **Object** - An object label [We do not use this]


In [ ]:
import pandas as pd


def read_single_qcinv_file(file_path):
    """Read in a single qcinv file and produce a pandas data frame of the information.

    Parameters
    ----------
    file_path : str
        The path to the file to read.

    Returns
    -------
    pd.Dataframe
        The data from the file.
    """
    # Precompute the ranges for each of the float columns we need to extract.
    _float_col_ranges = {
        "ra": (6, 12),
        "dec": (12, 18),
        "exptime": (28, 32),
        "secz": (32, 37),
        "psf": (37, 43),
        "sky": (43, 49),
        "cloud": (49, 55),
        "teff": (55, 61),
    }

    data = {
        "time": [],
        "ra": [],
        "dec": [],
        "exptime": [],
        "filter": [],
        "secz": [],
        "psf": [],
        "sky": [],
        "cloud": [],
        "teff": [],
    }

    with open(file_path, "r") as f:
        for line in f:
            # Skip comment lines, blank lines, and the line at the end that gives the MJD.
            if line.startswith("#") or line.strip() == "" or line.startswith("MJD"):
                continue

            # Extract the time and filter (two string columns).
            data["time"].append(mjd_from_filename(file_path, line[19:24].strip()))
            data["filter"].append(line[24:28].strip())

            # Extract all the float columns, handling blank spaces as NaN.
            for col_name, (start, end) in _float_col_ranges.items():
                value_str = line[start:end].strip()
                try:
                    value = float(value_str)
                except ValueError:
                    value = np.nan
                data[col_name].append(value)

    return pd.DataFrame(data)

We can use our helper functions to read in all the files and combine them.

In [ ]:
data_frames = []
for file_path in filenames:
    try:
        df = read_single_qcinv_file(file_path)
        data_frames.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

full_data = pd.concat(data_frames, ignore_index=True)
print(full_data.head())